<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/Architecture_Adaptation/28_moe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

### References:
Mixture of Experts Explained - [Link](https://huggingface.co/blog/moe)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # router + experts
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
                ) for _ in range(num_experts)
        ])
        self.top_k = top_k
        self.num_experts = num_experts

    def forward(self, x):
        # route tokens to top-k experts
        top_k_scores, top_k_idx = self.router(x).topk(self.top_k, dim=-1)
        top_k_scores = torch.softmax(top_k_scores, dim=-1)

        out = torch.zeros_like(x)
        for i in range(self.top_k):
          expert_idx = top_k_idx[:, :, i]
          expert_score = top_k_scores[:, :, i].unsqueeze(-1)

          for j in range(self.num_experts):
            mask = (expert_idx == j)

            if mask.any():
              out[mask] += expert_score[mask] * self.experts[j](x[mask])

        return out

In [12]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.9ms)
  ✅ [2/4] Has router and experts (1.5ms)
  ✅ [3/4] Router logits shape (2.1ms)
  ✅ [4/4] Gradient flow (47.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (54.8ms total)
  Progress saved. Run status() to see your dashboard.

